# From a fitted model to deliverables

You've got a model you're happy with -- now what do you actually hand to
someone else? Two things: a one-page diagnostic summary, and a rating
table in the format a pricing/actuarial team already works with.


## Fit a model (same book as the earlier notebooks)

In [1]:
from insurabench.data.datasets import make_synthetic_two_table
from insurabench.data.policy_frame import PolicyFrame
from insurabench.model_selection import train_test_split_policy_frame
from insurabench.models.glm import GLMPricingModel

book = make_synthetic_two_table(n_policies=4000, renewals=True, seed=7)
pf = PolicyFrame.from_tables(
    book.policies, book.claims, book.policy_schema, book.claims_schema, verbose=False
)
train_pf, test_pf = train_test_split_policy_frame(pf, test_size=0.25, seed=7)

model = GLMPricingModel(family="poisson").fit_policy_frame(train_pf, "frequency")


## I want a one-page summary of how this model performs

`generate_model_card` pulls together D², Gini, calibration, a lift
summary, AIC/BIC (for a GLM), and the biggest rating-factor relativities
-- the things you'd otherwise hand-assemble every time.


In [2]:
from insurabench.reporting import generate_model_card
from IPython.display import Markdown, display

card = generate_model_card(model, "frequency", test_pf, train_pf=train_pf)
display(Markdown(card.to_markdown()))


# Model card: GLMPricingModel

- **Family:** 'poisson'
- **Target:** `frequency`
- **Policy-periods:** 2,017 · **Claims:** 389 · **Exposure:** 1,317.2

## Key metrics

| Metric | Value |
|---|---|
| Train D² | -0.0265 |
| Test D² | -0.0243 |
| Gini (normalized) | 0.1137 |
| Calibration index (~1.0 = well-calibrated) | 0.8349 |
| Lift ratio (top decile / bottom decile, actual) | 1.82x |
| AIC | 2,177.5 |
| BIC | 2,228.0 |

## Top rating-factor relativities

| feature | level | n_obs | exposure | observed | fitted | relativity |
|---|---|---|---|---|---|---|
| vehicle_power | (13.0, 14.0] | 189 | 124.2100 | 0.2898 | 0.3762 | 1.4945 |
| vehicle_power | (12.0, 13.0] | 195 | 121.4070 | 0.3212 | 0.3597 | 1.4289 |
| vehicle_power | (11.0, 12.0] | 193 | 122.8470 | 0.4314 | 0.3451 | 1.3711 |
| vehicle_power | (10.0, 11.0] | 178 | 113.9360 | 0.3599 | 0.3289 | 1.3067 |
| vehicle_power | (9.0, 10.0] | 149 | 96.4030 | 0.2697 | 0.3168 | 1.2587 |
| vehicle_power | (8.0, 9.0] | 171 | 113.4060 | 0.2469 | 0.3016 | 1.1983 |
| driver_age | (73.0, 79.0] | 199 | 128.6150 | 0.2099 | 0.2797 | 0.8554 |
| vehicle_power | (7.0, 8.0] | 175 | 118.8980 | 0.2019 | 0.2904 | 1.1538 |
| driver_age | (68.0, 73.0] | 179 | 121.7540 | 0.2628 | 0.2891 | 0.8843 |
| driver_age | (62.0, 68.0] | 195 | 125.9010 | 0.2780 | 0.2910 | 0.8901 |

Every field on `card` is also available directly if you want to use the
numbers programmatically rather than just reading the rendered page --
e.g. `card.gini`, `card.test_d2`, `card.top_relativities` (a DataFrame).


## I want to hand the rating factors to a pricing team

`export_rating_table` writes the model's rating factors out in the
spreadsheet format a pricing/actuarial team actually works with: one
sheet per rating factor, base level clearly flagged.


In [3]:
from insurabench.reporting import export_rating_table

y_pred = model.predict_policy_frame(test_pf)
path = export_rating_table(test_pf, "frequency", y_pred, "rating_table.xlsx")
print(f"Wrote {path}")


Wrote rating_table.xlsx


In [4]:
import pandas as pd

sheets = pd.read_excel(path, sheet_name=None)
print("Sheets:", list(sheets))
sheets["driver_age"]


Sheets: ['driver_age', 'region', 'vehicle_brand', 'vehicle_power']


,level,n_obs,exposure,observed,fitted,relativity,is_base
0,"(23.0, 30.0]",236,151.692,0.257100,0.326964,1.000000,True
1,"(36.0, 43.0]",214,141.385,0.360717,0.317456,0.970921,False
2,"(55.0, 62.0]",220,141.083,0.255169,0.294476,0.900639,False
3,"(17.999, 23.0]",204,135.968,0.507472,0.326509,0.998610,False
4,"(73.0, 79.0]",199,128.615,0.209929,0.279682,0.855392,False
5,"(49.0, 55.0]",195,128.529,0.233410,0.302954,0.926566,False
6,"(62.0, 68.0]",195,125.901,0.277996,0.291036,0.890116,False
7,"(43.0, 49.0]",192,123.934,0.298546,0.303518,0.928294,False
8,"(68.0, 73.0]",179,121.754,0.262825,0.289125,0.884272,False
9,"(30.0, 36.0]",183,118.329,0.278883,0.316263,0.967272,False


A `.csv` path instead of `.xlsx` gives you one long tidy table instead
of a per-factor workbook -- same underlying numbers, pick whichever
shape the person on the other end wants.
